# R3maJ Kaggle Notebook v4

Kaggle GPU training notebook for R3maJ. Downloads `serialized_replays.bin` from Google Drive with `gdown`, clones the repository, builds the trainer, and starts training.

**Kaggle setup:** enable a GPU accelerator and Internet access in the notebook settings.

In [ ]:
# 1. Clone R3maJ
import os, subprocess
ROOT = '/kaggle/working/R3maJ'
REPO = 'https://github.com/vfxjamer/R3maJ.git'
if not os.path.isdir(os.path.join(ROOT, '.git')):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, ROOT], check=True)
else:
    subprocess.run(['git', '-C', ROOT, 'pull'], check=False)
print('ROOT:', ROOT)
print('contents:', sorted(os.listdir(ROOT)))

In [ ]:
# 2. Install gdown
!pip install -q gdown

import gdown, os
REPLAY_FILE_ID = 'YOUR_GOOGLE_DRIVE_FILE_ID'
LOCAL_REPLAY = '/kaggle/working/R3maJ/build/serialized_replays.bin'
os.makedirs(os.path.dirname(LOCAL_REPLAY), exist_ok=True)

if REPLAY_FILE_ID == 'YOUR_GOOGLE_DRIVE_FILE_ID':
    print('Set REPLAY_FILE_ID to your Google Drive file ID before running this cell.')
else:
    url = f'https://drive.google.com/uc?id={REPLAY_FILE_ID}'
    if not os.path.exists(LOCAL_REPLAY):
        print('Downloading serialized_replays.bin...')
        gdown.download(url, LOCAL_REPLAY, quiet=False)
    print('Replay exists:', os.path.exists(LOCAL_REPLAY))
    if os.path.exists(LOCAL_REPLAY):
        print('Replay size:', round(os.path.getsize(LOCAL_REPLAY) / (1024**3), 3), 'GB')

In [ ]:
# 3. Install build dependencies
import subprocess, os, sys
subprocess.run(['apt-get', 'update', '-qq'], check=False)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'build-essential', 'cmake', 'git', 'libpython3-dev', 'pkg-config'], check=False)
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA build:', torch.version.cuda)
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 4. Configure R3maJ
import os, subprocess, torch
os.chdir(ROOT)
torch_prefix = os.path.dirname(torch.__file__)
configure = ['cmake', '-S', '.', '-B', 'build', '-DCMAKE_BUILD_TYPE=Release', f'-DTORCH_INSTALL_PREFIX={torch_prefix}']
print(' '.join(configure))
r = subprocess.run(configure, text=True)
print('configure rc:', r.returncode)

In [ ]:
# 5. Build
import os, subprocess
nproc = os.cpu_count() or 2
print(f'Building with -j{nproc} ...')
r = subprocess.run(['cmake', '--build', 'build', '-j', str(nproc)], text=True)
print('build rc:', r.returncode)
EXE = os.path.join(ROOT, 'build', 'R3maJ')
print('binary exists:', os.path.exists(EXE), EXE)

In [ ]:
# 6. Verify replay + binary before training
import os
assert os.path.exists(EXE), 'R3maJ binary was not built.'
assert os.path.exists(LOCAL_REPLAY), 'serialized_replays.bin is missing. Set REPLAY_FILE_ID and rerun the download cell.'
print('READY FOR TRAINING')
print('binary:', EXE)
print('replays:', LOCAL_REPLAY)

In [ ]:
# 7. Start R3maJ training
# Edit TRAIN_ARGS if you want different games/checkpoint settings.
import os, subprocess
os.chdir(os.path.join(ROOT, 'build'))
TRAIN_ARGS = ['--device', 'cuda', '--save-dir', 'checkpoints', '--games', '164', '--replays', 'serialized_replays.bin']
print('Launching:', './R3maJ', *TRAIN_ARGS)
proc = subprocess.Popen(['./R3maJ'] + TRAIN_ARGS)
print('Training PID:', proc.pid)

## Notes
- This v4 notebook is Kaggle-specific; it does not use `google.colab.drive`.
- `serialized_replays.bin` is downloaded directly into the R3maJ build directory.
- Kaggle runtime storage is temporary. For long training, checkpoint persistence must be handled separately (for example, by Kaggle output/dataset storage or another external backup mechanism).
- Set the Google Drive replay file to an accessible sharing mode before using the simple `gdown` file-ID method.